In [37]:
# Library imports
import torch
import math
import random
import json
import nltk
import itertools
import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity
from external.claimify.src.claimify import Claimify

In [38]:
# Determine what device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(device)

    print(f'CUDA on {device_name}')
else:
    print(f'CPU')

CUDA on NVIDIA GeForce RTX 4070 Ti


In [39]:
# Check if everything's cleaned
allocated = torch.cuda.memory_allocated() / (1024 * 1024)
reserved = torch.cuda.memory_reserved() / (1024 * 1024)

print(f'Allocated: {allocated:.1f} MB; Reserved: {reserved:.1f} MB')

Allocated: 2463.4 MB; Reserved: 8598.0 MB


In [40]:
NLI_MODEL = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli' # https://huggingface.co/MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2' # https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

CLAIM_MODEL = "Qwen/Qwen3.5-0.8B"
CLAIM_MODEL_DTYPE = torch.float16

NLTK_MODEL = 'tokenizers/punkt/english.pickle'

In [41]:
#nltk.download('punkt_tab')

In [42]:
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)

claim_tokenizer = AutoTokenizer.from_pretrained(CLAIM_MODEL)
claim_model = AutoModelForCausalLM.from_pretrained(CLAIM_MODEL, dtype=CLAIM_MODEL_DTYPE).to(device)

embed_model = SentenceTransformer(EMBED_MODEL).to(device)

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

In [ ]:
dataset = json.load(open('datasets/ContraDoc/ContraDoc.json', 'r'))

print(f'Dataset loaded with {len(dataset["pos"])} positive and {len(dataset["neg"])} negative examples.')

Dataset loaded with 449 positive and 442 negative examples.


In [ ]:

def get_closest_sentence_pairs(sentences, embeddings, top_k=None):
    """
    Given a list of sentences and their embeddings, return sentence pairs with closest embeddings.
    
    Args:
        sentences: list of sentence strings
        embeddings: numpy array of embeddings for each sentence
        top_k: number of closest pairs to return
    
    Returns:
        list of tuples (sentence_a, sentence_b, similarity_score)
    """
    
    # Compute pairwise cosine similarity.
    similarity_matrix = cosine_similarity(embeddings)
    
    # Exclude self-similarity by setting diagonal to -1.
    for i in range(len(sentences)):
        similarity_matrix[i][i] = -1.0

    # For each sentence, keep only the most similar partner.
    pairs = [(i, np.argmax(similarity_matrix[i])) for i in range(len(sentences))]
    
    # Optionally keep only the highest-scoring pairs globally.
    if top_k is not None:
        pairs.sort(key=lambda x: similarity_matrix[x[0]][x[1]], reverse=True)
        pairs = pairs[:top_k]
        
    return [(sentences[i], sentences[j], similarity_matrix[i][j]) for i, j in pairs]

def predict(premise, hypothesis):
    """
    Run NLI classification for a (premise, hypothesis) pair.

    Returns probabilities for entailment, neutral, and contradiction.
    """
    # Tokenize both texts and move tensors to the selected device.
    model_inputs = nli_tokenizer(premise, hypothesis, truncation=True, return_tensors="pt").to(device)

    # Forward pass and probability conversion.
    output = nli_model(model_inputs["input_ids"] )
    prediction = torch.softmax(output["logits"][0], -1).tolist()
    label_names = ["entailment", "neutral", "contradiction"]

    return {name: float(pred) for pred, name in zip(prediction, label_names)}
    

def claim_extractor(prompt, temperature):
    """
    Generate claim-style text from a prompt using the configured causal LLM.

    Args:
        prompt: Instruction and source text for claim extraction.
        temperature: Decoding temperature (currently not used in generate()).

    Returns:
        Decoded generated text after optional <think> stripping.
    """
    # Build chat-format input expected by Qwen tokenizer/template.
    messages = [
        {"role": "user", "content": prompt}
    ]
    text = claim_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False  # Force concise non-thinking mode when supported.
    )
    model_inputs = claim_tokenizer([text], return_tensors="pt").to(device)

    # Generate continuation tokens.
    generated_ids = claim_model.generate(
        **model_inputs,
        max_new_tokens=32768
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    # If model emits a </think> marker token, skip internal reasoning part.
    try:
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    content = claim_tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    print(f'LLM:')
    print(f'Prompt:\n{prompt}\n')
    print(f'Response:\n{content}\n')

    return content

def extract_claims(question, answer):
    """
    Run the full Claimify pipeline for a question/answer pair and print claims.
    """
    c = Claimify(llm_function=claim_extractor)
    claims = c.extract_claims(question=question, answer=answer)
    print(claims)

def detect(document):
    """
    Detect potential contradictions inside one document.

    Steps:
    1. Split document into sentences (for diagnostics).
    2. Extract atomic claims with the claim model.
    3. Embed claims and pair each claim with a nearest neighbor.
    4. Score each pair with NLI and collect high-contradiction evidence.

    Returns:
        (max_contradiction_probability, evidence_list)
    """
    sentences = nltk.tokenize.sent_tokenize(document, language='english')
    sentence_count = len(sentences)

    print(f'Split text into {sentence_count} sentences.')

    prompt = 'Extract all facts from the following text. Respond with only the facts, with one fact per line. Each fact should be self-contained. Remove all pronouns and replace with proper nouns of the entity they refer to.'
    claims = claim_extractor(f'{prompt}\n\nText:\n{document}', 1.0).split('\n')

    print(f'Extracted {len(claims)} claims:')
    print(claims)

    # Embed claims for nearest-neighbor pairing.
    embeddings = embed_model.encode(claims)
    pairs = get_closest_sentence_pairs(claims, embeddings)

    p_contra_max = 0.0
    evidence = []

    # Evaluate each pair with NLI and track strongest contradiction signal.
    for a, b, similarity in tqdm.tqdm(pairs):
        prediction = predict(a, b)
        p_contra = prediction['contradiction']
        p_contra_max = max(p_contra_max, p_contra)

        print(f'Comparing:')
        print(f'  {a}')
        print(f'  {b}')
        print(f'  Similarity: {similarity:.4f}')
        print(f'  Prediction: {p_contra:.4f}')
        print()

        # Keep human-readable evidence above a fixed contradiction threshold.
        if p_contra > 0.75:
            evidence.append((a, b, p_contra))

    return p_contra_max, evidence

In [ ]:
premise = "The sky is red"
hypothesis = "The man is walking his dog under the blue sky"

print(predict(premise, hypothesis))

{'entailment': 5.8710575103759766e-05, 'neutral': 0.00026726722717285156, 'contradiction': 0.99951171875}


In [ ]:
pos = True					# Test example from positive or negative set
example_id = None			# Specific example ID to test, or None for random

# 3489738232_6
# story_train_3585

examples = dataset['pos' if pos else 'neg']

if example_id is not None:
	example = examples[example_id]
else:
	examples = list(examples.values())
	example = examples[random.randint(0, len(examples) - 1)]

print(f'Detecting contradictions in example {example["unique id"]}')
print()

print(example['text'])
print()

if pos:
	print(f'Evidence: {example["evidence"]}')
	print(f'Ref sentences: {example.get("ref sentences", [])}')

print()

p_contra, evidence = detect(example['text'])
p_contra, evidence

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


Detecting contradictions in example 3499318685_1

= Zrinski Battalion =

The Zrinski Battalion ( Croatian : Bojna Zrinski ) was a special forces unit of the Croatian National Guard ( Zbor narodne garde – ZNG ) and later of the Croatian Army ( Hrvatska vojska – HV ) established in Kumrovec on 18 May 1991 , during the Croatian War of Independence. The unit drew personnel from the special police forces and a former French Foreign Legion troops serving as its core. The battalion was set up and initially commanded by Ante Roso , while Major Miljenko Filipović took over as the commanding officer in August.   The Zrinski Battalion trained volunteer troops in Vukovar in June 1991 before it saw action in Hrvatska Kostajnica , the Battle of Gospić and near Slano in 1991. By the end of 1991 , the unit 's personnel were tasked with setting up an additional special forces unit of the HV. The next year its elements took part in the Battle of Kupres and Operation Tiger aimed at lifting the Siege of D

 30%|███       | 3/10 [00:00<00:00, 24.80it/s]

Comparing:
  - The Zrinski Battalion (Croatian: Bojna Zrinski) was a special forces unit of the Croatian National Guard (Zbor narodne garde – ZNG) and later of the Croatian Army (Hrvatska vojska – HV).
  - In February 1994, the Zrinski Battalion was amalgamated with several other HV special forces units into the 1st Croatian Guards Brigade (1. hrvatski gardijski zdrug), a component of the 1st Croatian Guards Corps (1. hrvatski gardijski zbor).
  Similarity: 0.8464
  Prediction: 0.0004

Comparing:
  - The unit was established in Kumrovec on 18 May 1991.
  - By the end of 1991, the unit's personnel were tasked with setting up an additional special forces unit of the HV.
  Similarity: 0.3900
  Prediction: 0.0003

Comparing:
  - The unit drew personnel from the special police forces and a former French Foreign Legion troops serving as its core.
  - By the end of 1991, the unit's personnel were tasked with setting up an additional special forces unit of the HV.
  Similarity: 0.4802
  Predic

 60%|██████    | 6/10 [00:00<00:00, 24.90it/s]

Comparing:
  - The Zrinski Battalion trained volunteer troops in Vukovar in June 1991 before it saw action in Hrvatska Kostajnica, the Battle of Gospić and near Slano in 1991.
  - In February 1994, the Zrinski Battalion was amalgamated with several other HV special forces units into the 1st Croatian Guards Brigade (1. hrvatski gardijski zdrug), a component of the 1st Croatian Guards Corps (1. hrvatski gardijski zbor).
  Similarity: 0.6590
  Prediction: 0.0006

Comparing:
  - By the end of 1991, the unit's personnel were tasked with setting up an additional special forces unit of the HV.
  - In February 1994, the Zrinski Battalion was amalgamated with several other HV special forces units into the 1st Croatian Guards Brigade (1. hrvatski gardijski zdrug), a component of the 1st Croatian Guards Corps (1. hrvatski gardijski zbor).
  Similarity: 0.4812
  Prediction: 0.0060



100%|██████████| 10/10 [00:00<00:00, 25.92it/s]

Comparing:
  - The next year its elements took part in the Battle of Kupres and Operation Tiger aimed at lifting the Siege of Dubrovnik.
  - In 1993, the battalion took part in Operation Maslenica.
  Similarity: 0.3888
  Prediction: 0.0157

Comparing:
  - It also helped develop and train the Croatian Defence Council (Hrvatsko vijeće obrane – HVO), setting up a training camp in Tomislavgrad.
  - The Zrinski Battalion (Croatian: Bojna Zrinski) was a special forces unit of the Croatian National Guard (Zbor narodne garde – ZNG) and later of the Croatian Army (Hrvatska vojska – HV).
  Similarity: 0.5518
  Prediction: 0.0013

Comparing:
  - In 1993, the battalion took part in Operation Maslenica.
  - The battalion was initially commanded by Ante Roso, while Major Miljenko Filipović took over as the commanding officer in August.
  Similarity: 0.5526
  Prediction: 0.0002

Comparing:
  - In February 1994, the Zrinski Battalion was amalgamated with several other HV special forces units into the 

(0.46923828125, [])